# Multimodal Human Activity Recognition on MMAct (Kaggle GPU Pipeline)
## Improved Custom CNN vs. MobileNetV2 Transfer Learning Benchmark

This notebook executes the complete Multimodal Human Activity Recognition (HAR) pipeline on the **MMAct dataset**, comparing an **Improved Custom CNN** (with spatial dropout, batch normalization, and weight decay to eliminate the prior 97% train vs. 57% test overfitting gap) against **MobileNetV2** (leveraging transfer learning with ImageNet pre-trained representations) on identical multimodal splits and 10 verified activity classes:
`crouching`, `jumping`, `kicking`, `pointing`, `running`, `sitting`, `standing`, `using_phone`, `walking`, `waving_hand`.

---
### Kaggle Setup Instructions:
1. **Enable GPU Accelerator:**
   - In the right-hand panel under **Session Options** -> **Accelerator**, select **GPU T4 x2** or **GPU P100**.
2. **Enable Internet Access:**
   - In the right-hand panel under **Session Options** -> **Internet**, toggle to **On** (required for GitHub sync and pretrained weights).
3. **Attach MMAct Dataset:**
   - Click **+ Add Input** in the top right.
   - Search for and attach your uploaded dataset (e.g., `mmact-trimmed` containing extracted video/sensors or nested ZIP archives).
   - The notebook automatically scans `/kaggle/input`, detects extracted directories or nested ZIP archives, prepares the data in `/kaggle/working/HAR-MMAct/data/genuine_mmact/`, and runs pre-flight validation.

In [ ]:
# Step 1: GPU & PyTorch Environment Verification
!nvidia-smi

import torch
print('=' * 70)
print('PyTorch Version   :', torch.__version__)
print('CUDA Available    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name   :', torch.cuda.get_device_name(0))
    print('Device Count      :', torch.cuda.device_count())
    print('Compute Capability:', torch.cuda.get_device_capability(0))
else:
    print('[WARNING] CUDA GPU is not detected! Ensure GPU Accelerator is enabled in Kaggle Settings.')
print('=' * 70)


In [ ]:
# Step 2: Clone/Update Repository, Set sys.path, and Install Dependencies
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
REPO_URL = 'https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git'

# 1. Clone or update repository into PROJECT_ROOT
if not PROJECT_ROOT.exists():
    print(f'Cloning {REPO_URL} into {PROJECT_ROOT}...')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f'Repository already present at {PROJECT_ROOT}. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=False)

# 2. Configure import path before importing project modules
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT configured : {PROJECT_ROOT}')
print(f'sys.path[0] set to      : {sys.path[0]}')

# 3. Inspect dependency configuration and install safely
req_candidates = list(PROJECT_ROOT.glob('*requirements*.txt'))
pyproject_file = PROJECT_ROOT / 'pyproject.toml'

if req_candidates:
    req_file = req_candidates[0]
    print(f'Installing dependencies from {req_file.name} (using --prefer-binary to protect Kaggle runtime)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '-r', str(req_file)], check=False)
elif pyproject_file.exists():
    print(f'Installing project from {pyproject_file.name}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '-e', str(PROJECT_ROOT)], check=False)
else:
    print('No requirements file found; using preinstalled Kaggle libraries.')

# 4. Verify project imports
try:
    from src.data.ingestion import DatasetIngestionPipeline
    from src.data.indexer import MMActIndexer
    from src.models.fusion_model import MultimodalHARModel
    from src.utils.plots import plot_cnn_vs_mobilenet_comparison, plot_per_class_metrics
    print('SUCCESS: Verified project imports (DatasetIngestionPipeline, MMActIndexer, MultimodalHARModel).')
except ImportError as e:
    raise ImportError(f'Failed to import project modules from {PROJECT_ROOT}: {e}')


In [ ]:
# Step 3: Deep Inspection of /kaggle/input & Robust Dataset Ingestion
import os
import shutil
from pathlib import Path
from src.data.ingestion import DatasetIngestionPipeline

kaggle_input = Path('/kaggle/input')
target_data_dir = PROJECT_ROOT / 'data' / 'genuine_mmact'
target_trimmed = target_data_dir / 'trimmed'

# Clean any stale cache or broken symlinks from prior runs
if target_trimmed.exists() and not target_trimmed.is_dir():
    target_trimmed.unlink()
target_trimmed.mkdir(parents=True, exist_ok=True)

print('=' * 70)
print('DEEP INSPECTION OF /kaggle/input')
print('=' * 70)

all_input_items = list(kaggle_input.glob('*')) if kaggle_input.exists() else []
print(f'Found {len(all_input_items)} top-level dataset entries in /kaggle/input:')
for it in all_input_items:
    print(f'  - {it.name} ({ "Directory" if it.is_dir() else "File" })')

# Print directory tree up to depth 3
def print_tree(dir_path: Path, max_depth: int = 3, current_depth: int = 0):
    if current_depth > max_depth or not dir_path.is_dir():
        return
    items = sorted(list(dir_path.iterdir()))
    for item in items[:8]:  # Show first 8 items per folder to avoid spam
        indent = '    ' * current_depth
        print(f'{indent}|-- {item.name}')
        if item.is_dir() and current_depth < max_depth:
            print_tree(item, max_depth, current_depth + 1)
    if len(items) > 8:
        indent = '    ' * current_depth
        print(f'{indent}|-- ... ({len(items) - 8} more items)')

for it in all_input_items:
    print(f'\nDirectory tree for: {it.name}')
    print_tree(it, max_depth=3)

# 1. Search for extracted video and sensor directories
detected_video_dir = None
detected_sensor_dir = None
detected_split_dir = None
searched_paths = []

search_candidates = all_input_items if all_input_items else [PROJECT_ROOT / 'data']
# Prioritize directories matching mmact
search_candidates = sorted(search_candidates, key=lambda d: ('mmact' not in d.name.lower(), d.name))

for d in search_candidates:
    searched_paths.append(str(d))
    # Candidate video paths
    cand_videos = [
        d / 'trimmed' / 'video',
        d / 'video',
        d / 'genuine_mmact' / 'trimmed' / 'video',
        d / 'data' / 'genuine_mmact' / 'trimmed' / 'video',
        d / 'mmact-trimmed' / 'trimmed' / 'video',
        d / 'mmact-trimmed' / 'video',
    ]
    for c in cand_videos:
        if c.is_dir() and any(c.rglob('*.mp4')):
            detected_video_dir = c
            break
    if detected_video_dir:
        break

    # Fallback recursive search for mp4 files
    mp4_samples = list(d.rglob('*.mp4'))
    if mp4_samples:
        for parent in mp4_samples[0].parents:
            if parent.name == 'video' or (parent.parent and parent.parent.name == 'video'):
                detected_video_dir = parent if parent.name == 'video' else parent.parent
                break
        if not detected_video_dir:
            detected_video_dir = mp4_samples[0].parent.parent.parent.parent
        break

for d in search_candidates:
    cand_sensors = [
        d / 'trimmed' / 'sensors',
        d / 'sensors',
        d / 'genuine_mmact' / 'trimmed' / 'sensors',
        d / 'data' / 'genuine_mmact' / 'trimmed' / 'sensors',
        d / 'mmact-trimmed' / 'trimmed' / 'sensors',
        d / 'mmact-trimmed' / 'sensors',
    ]
    for c in cand_sensors:
        if c.is_dir() and any(c.rglob('*.csv')):
            detected_sensor_dir = c
            break
    if detected_sensor_dir:
        break

    csv_samples = list(d.rglob('*.csv'))
    if csv_samples:
        for parent in csv_samples[0].parents:
            if parent.name == 'sensors' or parent.name in ['acc_phone_clip', 'acc_watch_clip', 'gyro_clip']:
                detected_sensor_dir = parent if parent.name == 'sensors' else parent.parent
                break
        if not detected_sensor_dir:
            detected_sensor_dir = csv_samples[0].parent.parent.parent
        break

for d in search_candidates:
    cand_splits = [
        d / 'trimmed' / 'splits',
        d / 'splits',
        d / 'genuine_mmact' / 'trimmed' / 'splits',
        d / 'data' / 'genuine_mmact' / 'trimmed' / 'splits',
        d / 'mmact-trimmed' / 'trimmed' / 'splits',
    ]
    for c in cand_splits:
        if c.is_dir() and any(c.glob('*.txt')):
            detected_split_dir = c
            break
    if detected_split_dir:
        break
    split_txts = list(d.rglob('*trimmed*x-*.txt')) or list(d.rglob('*train*.txt'))
    if split_txts:
        detected_split_dir = split_txts[0].parent
        break

# 2. Search for ZIP archives (supporting nested ZIPs)
detected_zips = []
for d in search_candidates:
    zips = list(d.rglob('*.zip'))
    detected_zips.extend(zips)

print('\n' + '=' * 70)
print('DATASET DISCOVERY OUTCOME:')
print(f'  - Detected Video Directory  : {detected_video_dir}')
print(f'  - Detected Sensor Directory : {detected_sensor_dir}')
print(f'  - Detected Split Directory  : {detected_split_dir}')
print(f'  - Detected ZIP Archives     : {len(detected_zips)} archives')
for z in detected_zips[:4]:
    print(f'      * {z.relative_to(kaggle_input if kaggle_input.exists() else PROJECT_ROOT)}')
print('=' * 70)

# Helper to link or copy
def link_or_copy(src: Path, dst: Path):
    if dst.exists():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    try:
        os.symlink(src.resolve(), dst)
        print(f'Symlinked: {src} -> {dst}')
    except Exception as e:
        print(f'Symlink unavailable ({e}), copying directory tree...')
        shutil.copytree(src.resolve(), dst, dirs_exist_ok=True)

if detected_video_dir and detected_sensor_dir:
    print('Using extracted MMAct directory structure...')
    dest_video = target_trimmed / 'video'
    dest_sensors = target_trimmed / 'sensors'
    dest_splits = target_trimmed / 'splits'
    link_or_copy(detected_video_dir, dest_video)
    link_or_copy(detected_sensor_dir, dest_sensors)
    if detected_split_dir:
        link_or_copy(detected_split_dir, dest_splits)

elif detected_zips:
    print(f'Unpacking {len(detected_zips)} nested/direct ZIP archives via DatasetIngestionPipeline...')
    dest_downloads = PROJECT_ROOT / 'data' / 'downloads'
    dest_downloads.mkdir(parents=True, exist_ok=True)
    for z in detected_zips:
        tgt_z = dest_downloads / z.name
        if not tgt_z.exists():
            try:
                os.symlink(z.resolve(), tgt_z)
            except Exception:
                shutil.copy(z.resolve(), tgt_z)

    ingestion = DatasetIngestionPipeline(
        incoming_dir=dest_downloads,
        raw_storage_dir=target_trimmed,
    )
    records = ingestion.scan_and_ingest()
    print(f'Ingestion complete. Processed {len(records)} archives/files.')
else:
    error_msg = (
        f'Could not find valid MMAct video/sensor folders or ZIP archives in /kaggle/input.\n'
        f'Searched paths:\n' + '\n'.join([f'  - {p}' for p in searched_paths]) + '\n'
        f'Please attach your MMAct dataset (e.g. mmact-trimmed) in the Kaggle right-hand panel (+ Add Input).'
    )
    raise FileNotFoundError(error_msg)


In [ ]:
# Step 4: Pre-Flight Validation & Split Isolation Check
from src.data.indexer import MMActIndexer
from src.data.splits import DatasetSplitter, verify_split_isolation
from src.config import SplitConfig, VERIFIED_TARGET_CLASSES

print('=' * 70)
print('PRE-FLIGHT VALIDATION & DIAGNOSTICS')
print('=' * 70)
print(f'Repository Root : {PROJECT_ROOT}')
print(f'Dataset Root    : {target_data_dir.resolve()}')

video_files = list(target_data_dir.rglob('*.mp4'))
sensor_files = list(target_data_dir.rglob('*.csv'))
split_files = list(target_data_dir.rglob('*.txt'))

print(f'\n[1] Video Modality:')
print(f'    - Total .mp4 files found : {len(video_files)}')
if video_files:
    print(f'    - Representative path    : {video_files[0].relative_to(target_data_dir)}')

print(f'\n[2] Wearable Sensor Modality:')
print(f'    - Total .csv files found : {len(sensor_files)}')
if sensor_files:
    print(f'    - Representative path    : {sensor_files[0].relative_to(target_data_dir)}')

print(f'\n[3] Split Definitions:')
print(f'    - Total split files found: {len(split_files)}')
for sf in split_files[:4]:
    print(f'    - Split file: {sf.name}')

# Indexer pairing and class coverage
indexer = MMActIndexer(dataset_root=target_data_dir)
report = indexer.index()

print(f'\n[4] Multimodal Sample Pairing:')
print(f'    - Paired Multimodal Samples: {report.paired_samples_count}')
print(f'    - Target Class Coverage ({len(report.class_distribution)}/10 classes):')
for cls_name in VERIFIED_TARGET_CLASSES:
    print(f'        * {cls_name:<15}: {report.class_distribution.get(cls_name, 0)} samples')

# Split verification
splitter = DatasetSplitter(SplitConfig(split_strategy='subject'))
train_samples, val_samples, test_samples = splitter.split(report.paired_samples)
isolation = verify_split_isolation(train_samples, val_samples, test_samples, strategy='subject')

print(f'\n[5] Subject-Independent Split Composition:')
print(f'    - Train Set : {len(train_samples)} samples')
print(f'    - Val Set   : {len(val_samples)} samples')
print(f'    - Test Set  : {len(test_samples)} samples')
print(f'    - Split Leakage Verification:')
for check_k, passed in isolation.items():
    print(f'        * {check_k:<25}: {"PASSED (Zero Leakage)" if passed else "FAILED"}')
    assert passed, f'CRITICAL: Split leakage detected on {check_k}!'

# Strict assertions
assert len(video_files) > 0, 'No .mp4 video files found!'
assert len(sensor_files) > 0, 'No .csv sensor files found!'
assert report.paired_samples_count > 0, 'Zero paired multimodal samples!'
print('\n>>> PRE-FLIGHT VALIDATION PASSED: Genuine dataset verified with zero split leakage.')
print('=' * 70)


In [ ]:
# Step 5: Train Model A - Improved Custom CNN (Anti-Overfitting Regularization)
# Configurable epoch count: set to 50 for full training or 3 for fast benchmark
EPOCHS = 50

print(f'Training Improved Custom CNN for {EPOCHS} epochs...')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --video_backbone cnn \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.001 \
    --split_strategy subject \
    --early_stopping_patience 12 \
    --checkpoint_name best_model_cnn.pth \
    --output_dir outputs \
    --cache_dir outputs/cache


In [ ]:
# Step 6: Train Model B - MobileNetV2 (Transfer Learning with Pretrained Weights)
print(f'Training MobileNetV2 for {EPOCHS} epochs...')
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --video_backbone mobilenet_v2 \
    --sensor_backbone inception_time \
    --epochs {EPOCHS} \
    --batch_size 8 \
    --lr 0.0001 \
    --weight_decay 0.0005 \
    --split_strategy subject \
    --early_stopping_patience 12 \
    --checkpoint_name best_model_mobilenet_v2.pth \
    --output_dir outputs \
    --cache_dir outputs/cache


In [ ]:
# Step 7: Standalone Evaluation on Held-Out Test Set (Both Models)
print('Evaluating Model A (Improved Custom CNN) on held-out test split...')
!python scripts/evaluate.py \
    --checkpoint outputs/checkpoints/best_model_cnn.pth \
    --data_root data/genuine_mmact \
    --output_file outputs/metrics_cnn.json \
    --plots_dir outputs/plots

print('\nEvaluating Model B (MobileNetV2) on held-out test split...')
!python scripts/evaluate.py \
    --checkpoint outputs/checkpoints/best_model_mobilenet_v2.pth \
    --data_root data/genuine_mmact \
    --output_file outputs/metrics_mobilenet_v2.json \
    --plots_dir outputs/plots

# Load and display model comparison summary
import json
with open('outputs/metrics_cnn.json') as f:
    m_cnn = json.load(f)
with open('outputs/metrics_mobilenet_v2.json') as f:
    m_mob = json.load(f)

print('\n' + '=' * 80)
print(' HELD-OUT TEST EVALUATION COMPARISON (SUBJECT-INDEPENDENT SPLIT)')
print('=' * 80)
print(f'{"Model Architecture":<25} | {"Test Accuracy":<14} | {"Macro F1":<12} | {"Macro Prec":<12} | {"Macro Rec":<12}')
print('-' * 80)
print(f'{"Custom CNN (Regularized)":<25} | {m_cnn["multimodal"]["accuracy"]*100:6.2f}%       | {m_cnn["multimodal"]["macro_f1"]:8.4f}   | {m_cnn["multimodal"]["macro_precision"]:8.4f}   | {m_cnn["multimodal"]["macro_recall"]:8.4f}')
print(f'{"MobileNetV2 (Transfer)":<25} | {m_mob["multimodal"]["accuracy"]*100:6.2f}%       | {m_mob["multimodal"]["macro_f1"]:8.4f}   | {m_mob["multimodal"]["macro_precision"]:8.4f}   | {m_mob["multimodal"]["macro_recall"]:8.4f}')
print('=' * 80)
print('\nPrior baseline discrepancy note:')
print('The student\'s prior unregularized run exhibited severe overfitting (97% train vs 57% test).')
print('These measured test results reflect actual outputs on the strict subject-independent split.')


In [ ]:
# Step 8: Generate & Display Visualizations
import json
from IPython.display import Image, display
from src.utils.plots import plot_cnn_vs_mobilenet_comparison

# Generate the side-by-side comparison plot from actual histories
hist_cnn_path = Path('outputs/history_cnn.json')
hist_mob_path = Path('outputs/history_mobilenet_v2.json')

if hist_cnn_path.exists() and hist_mob_path.exists():
    with open(hist_cnn_path) as f:
        h_cnn = json.load(f)
    with open(hist_mob_path) as f:
        h_mob = json.load(f)
    plot_cnn_vs_mobilenet_comparison(
        h_cnn,
        h_mob,
        'outputs/plots/cnn_vs_mobilenet_comparison.png',
        split_strategy='subject-independent',
    )
    print('Generated outputs/plots/cnn_vs_mobilenet_comparison.png')

plots_to_render = [
    ('CNN vs MobileNetV2 Accuracy Trajectory Comparison', 'outputs/plots/cnn_vs_mobilenet_comparison.png'),
    ('Custom CNN: Train vs Validation Loss', 'outputs/plots/train_vs_val_loss_cnn.png'),
    ('Custom CNN: Train vs Validation Accuracy', 'outputs/plots/train_vs_val_accuracy_cnn.png'),
    ('Custom CNN: Test Confusion Matrix', 'outputs/plots/test_confusion_matrix_cnn.png'),
    ('Custom CNN: Per-Class Precision, Recall, F1', 'outputs/plots/test_per_class_cnn.png'),
    ('Custom CNN: Test ROC Curves (OvR)', 'outputs/plots/test_roc_ovr_cnn.png'),
    ('MobileNetV2: Train vs Validation Loss', 'outputs/plots/train_vs_val_loss_mobilenet_v2.png'),
    ('MobileNetV2: Train vs Validation Accuracy', 'outputs/plots/train_vs_val_accuracy_mobilenet_v2.png'),
    ('MobileNetV2: Test Confusion Matrix', 'outputs/plots/test_confusion_matrix_mobilenet_v2.png'),
    ('MobileNetV2: Per-Class Precision, Recall, F1', 'outputs/plots/test_per_class_mobilenet_v2.png'),
    ('MobileNetV2: Test ROC Curves (OvR)', 'outputs/plots/test_roc_ovr_mobilenet_v2.png'),
]

for title, p in plots_to_render:
    if Path(p).exists():
        print('\n' + '=' * 70)
        print(title)
        print('=' * 70)
        display(Image(filename=p))


In [ ]:
# Step 9: Sample Multimodal Inference Demo
sample_video = 'data/genuine_mmact/trimmed/video/cross_scene/trainval/subject16/cam1/scene1/session1/crouching.mp4'
sample_phone = 'data/genuine_mmact/trimmed/sensors/acc_phone_clip/subject16/scene1/session1/crouching.csv'
sample_watch = 'data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject16/scene1/session1/crouching.csv'
sample_gyro  = 'data/genuine_mmact/trimmed/sensors/gyro_clip/subject16/scene1/session1/crouching.csv'

print('Running inference with Model A (Improved Custom CNN)...')
!python scripts/infer.py \
    --video {sample_video} \
    --sensor_phone {sample_phone} \
    --sensor_watch {sample_watch} \
    --sensor_gyro {sample_gyro} \
    --checkpoint outputs/checkpoints/best_model_cnn.pth

print('\nRunning inference with Model B (MobileNetV2)...')
!python scripts/infer.py \
    --video {sample_video} \
    --sensor_phone {sample_phone} \
    --sensor_watch {sample_watch} \
    --sensor_gyro {sample_gyro} \
    --checkpoint outputs/checkpoints/best_model_mobilenet_v2.pth


In [ ]:
# Step 10: Package All Artifacts for 1-Click Download
import shutil
from pathlib import Path

dest_outputs = Path('/kaggle/working/outputs')
if Path('outputs').resolve() != dest_outputs.resolve():
    shutil.copytree('outputs', dest_outputs, dirs_exist_ok=True)

# Archive all outputs into a single zip file
shutil.make_archive('/kaggle/working/mmact_benchmark_results', 'zip', 'outputs')
print('=' * 70)
print('All results, checkpoints, summaries, and plots successfully packaged!')
print('Download path: /kaggle/working/mmact_benchmark_results.zip')
print('Access via the Kaggle right-hand panel under the "Output" tab.')
print('=' * 70)
